In [ ]:
!pip install git+https://github.com/huggingface/transformers accelerate
!pip install qwen-vl-utils[decord]==0.0.8
!pip install flash_attn-2.7.4.post1+cu12torch2.6cxx11abiTRUE-cp39-cp39-linux_x86_64.whl



  Cloning https://github.com/huggingface/transformers to /tmp/pip-req-build-pedey_e6
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers /tmp/pip-req-build-pedey_e6
  Resolved https://github.com/huggingface/transformers to commit 9055ee4dd9ae6e258b8244faccfbdfa5c8e313e4
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 534.5/534.5 kB 41.6 MB/s eta 0:00:00
  Created wheel for transformers: filename=transformers-5.0.0.dev0-py3-none-any.whl size=11155169 sha256=ef07493510c10e48627b36aaf08567ff2037132fe95b44735b383f808e732339
  Stored in directory: /tmp/pip-ephem-wheel-cache-m11p0b_7/wheels/49/a7/50/c9fdabbf10e51bb1256adb0c1a587fedd7184f5bad28d47fe3
Successfully built transformers
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 0.36.0
    Uninstalling huggingface-hub-0.36.0:
 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from transformers import AutoProcessor, Glm4vForConditionalGeneration
import torch

MODEL_PATH = "zai-org/GLM-4.6V-Flash"
processor = AutoProcessor.from_pretrained(MODEL_PATH, use_fast=True)
model = Glm4vForConditionalGeneration.from_pretrained(
    pretrained_model_name_or_path=MODEL_PATH,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/367 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/20.0M [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/369 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/704 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [ ]:
import torch
from transformers import AutoProcessor, Glm4vForConditionalGeneration



def inference_glm_with_budget(model, processor, image_path, prompt, thinking_budget, max_new_tokens):
    """
    Runs inference with the GLM-4.6V model using a "thinking budget".

    This function performs a two-step generation to avoid reasoning loops.
    Returns only the model's final answer, excluding the thinking content.

    Args:
        model: The loaded Glm4vForConditionalGeneration model.
        processor: The AutoProcessor matching the model.
        image_path (str): Web URL or local path of the image.
        prompt (str): The text prompt for the model.
        thinking_budget (int): Maximum number of tokens for the reasoning phase.
        max_new_tokens (int): Maximum total number of tokens to generate.

    Returns:
        str: The model's final answer (text after </think>).
    """

    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "url": image_path
                },
                {
                    "type": "text",
                    "text": prompt
                }
            ],
        }
    ]

    inputs = processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors="pt"
    ).to(model.device)


    input_length = inputs["input_ids"].shape[1]

    generated_ids_step1 = model.generate(
        **inputs, max_new_tokens=thinking_budget, do_sample = False
    )
    output_ids_step1 = generated_ids_step1[0][input_length:].tolist()

    end_token_id = processor.tokenizer.eos_token_id
    end_think_token_id = processor.tokenizer.convert_tokens_to_ids('</think>')
    input_ids_for_second_gen_tensor = generated_ids_step1
    needs_second_generation = True

    if end_token_id in output_ids_step1:
        needs_second_generation = False
    elif end_think_token_id not in output_ids_step1:
        '''early_stopping_text = (
            "\n</think>\n"
            "FINAL ANSWER PROTOCOL ENGAGED.\n"
            "The analysis phase is complete. Output the most likely SPICE netlist based on the preceding analysis. "
            "Do not add further commentary, verification, or apologies, even if confidence is low. "
            "The output must be the netlist ONLY.\n"
        )'''
        early_stopping_text = (
          "\n</think>\n"
          "FINAL ANSWER PROTOCOL ENGAGED.\n"
          "The reasoning phase has been forcibly stopped due to token budget limits.\n"
          "Before producing the final output, perform an internal check for repetition or looping patterns "
          "in the preceding reasoning (e.g., repeated phrases, redundant chains, or self-referential reasoning). "
          "Ignore and discard any such repetitive or looping segments.\n"
          "Now, synthesize the final SPICE netlist using only the coherent and non-repetitive analytical content "
          "from the reasoning above.\n"
          "Do not include commentary or explanations. The output must be the SPICE netlist ONLY.\n"
      )
        early_stopping_ids = processor.tokenizer(
            early_stopping_text, return_tensors="pt", add_special_tokens=False
        ).input_ids.to(model.device)
        input_ids_for_second_gen_tensor = torch.cat([generated_ids_step1, early_stopping_ids], dim=-1)

    final_generated_ids = input_ids_for_second_gen_tensor
    if needs_second_generation:
        num_tokens_so_far = input_ids_for_second_gen_tensor.shape[1] - input_length
        remaining_tokens = max_new_tokens - num_tokens_so_far
        if remaining_tokens > 0:
            inputs_for_second_gen = inputs.copy()
            inputs_for_second_gen['input_ids'] = input_ids_for_second_gen_tensor
            inputs_for_second_gen['attention_mask'] = torch.ones_like(input_ids_for_second_gen_tensor)

            final_generated_ids = model.generate(
                **inputs_for_second_gen, # ** passes the whole updated dictionary
                max_new_tokens=remaining_tokens,
                do_sample = False,
                #repetition_penalty=1.3
            )

    # Debug print
    full_generated_text = processor.decode(final_generated_ids[0][input_length:], skip_special_tokens=True).strip()
    print("-" * 50)
    print(">>> DEBUG: FULL OUTPUT GENERATED BY THE MODEL <<<")
    print(full_generated_text)
    print("-" * 50)

    # Decode and parse the final answer
    final_output_ids = final_generated_ids[0][input_length:].tolist()
    try:
        index_end_think = len(final_output_ids) - final_output_ids[::-1].index(end_think_token_id)
    except ValueError:
        index_end_think = 0

    final_answer = processor.decode(
        final_output_ids[index_end_think:],
        skip_special_tokens=True
    ).strip()

    return final_answer


In [ ]:
import os, json
from PIL import Image

# Input image directory and output directory
image_directory = "/content/drive/MyDrive/netlist_test"
output_directory = "results/" # Directory where the text files are saved

# Make sure the output directory exists
os.makedirs(output_directory, exist_ok=True)

prompt = """
You are an expert electronics engineer specializing in the precise generation of topology-focused SPICE netlists from circuit schematics. Your analysis must be meticulous and follow a strict internal protocol to ensure accuracy.

**OUTPUT FORMAT:**
- The output MUST BE ONLY the SPICE netlist.
- Do NOT include any additional text, explanations, greetings, or markdown code blocks like ```spice ... ```.

---
**STRICT ANALYSIS PROTOCOL**
You MUST follow this internal thought process before and during generation.

**PHASE 1: Silent Global Net Identification**
- Before writing any output, first silently identify all distinct labeled connection points (nets) in the schematic. This is your internal reference map.
- Use '0' for the ground net (GND, VSS, or ground symbol).

**PHASE 2: Component-by-Component Netlist Generation**
- After your silent net identification, process each component one by one to generate its corresponding SPICE netlist line.

**Analysis Sub-Protocol for MOSFETs (CRITICAL)**
- Before writing the SPICE line for ANY MOSFET, you MUST perform this internal analysis:
    1.  **Identify Pins Geometrically:** Use this exact logic: The single pin on its own side is the **Gate**. Of the two pins on the other side, the one with an arrow is the **Source**. The remaining pin is the **Drain**.
    2.  **Map Pins to Nets:** Mentally map each identified pin (Gate, Source, Drain) to the correct net name from your Phase 1 analysis.
    3.  **Generate the SPICE Line:** Only after the previous two steps, write the SPICE line using the mapped nets in the STRICT order: `M# <Drain_Net> <Gate_Net> <Source_Net> <Bulk_Net> MODEL_TYPE`.

---
**GENERAL RULES TO APPLY:**
1.  **Component Naming:** Use standard initial letters (M for MOSFET, R for Resistor, etc.). Number them sequentially (M1, M2, R1, R2...).
2.  **Component Values:** Extract values (e.g., '1k') only if clearly visible. If not specified, use a placeholder like 'VALUE'.
3.  **MOSFET Bulk Connection:** If the schematic shows a 3-pin MOSFET (no explicit bulk connection), the Bulk (B) terminal MUST be connected to the SAME net as the Source (S).
4.  **EXCLUSIONS (Very Important):** DO NOT INCLUDE voltage sources (Vdd, Vin, Vb) or current sources. Do NOT generate `.MODEL`, `.OP`, `.END`, or `.SUBCKT` lines.

---
**EXAMPLE OF CORRECT EXECUTION:**
*   **If you see a schematic with:** A single NMOS transistor (M1) whose gate is connected to a net labeled 'Vin', source to ground ('0'), and drain to a net labeled 'Vout'. A resistor (R1) with value '1k' is connected between a net labeled 'VDD' and 'Vout'.
*   **The ONLY correct output is:**
M1 Vout Vin 0 0 NMOS
R1 VDD Vout 1k

---
Now, analyze the following image and generate the SPICE netlist by STRICTLY following the protocol above.
"""




for filename in os.listdir(image_directory):
    if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif')):
        image_path = os.path.join(image_directory, filename)
        print(image_path)

        response = inference_glm_with_budget(model, processor, image_path, prompt, 6000, 8192)
        print(response)



        base_filename = os.path.splitext(filename)[0]
        output_filename = f"{base_filename}.txt"
        output_path = os.path.join(output_directory, output_filename)

        with open(output_path, 'w', encoding='utf-8') as f:
            f.write(response)

        print(f"Response saved to: {output_path}\n")



/content/drive/MyDrive/netlist_test/Multivibrator.jpg


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


--------------------------------------------------
>>> DEBUG: FULL OUTPUT GENERATED BY THE MODEL <<<
<think>Got it, let's tackle this step by step.

First, Phase 1: Identify all distinct labeled connection points (nets). Looking at the schematic:

- The battery's negative terminal is ground, so that's net '0'.
- Then we have the nodes labeled with components: R1 is connected to a node, then L1, Q1's base? Wait, no, let's list all nodes:

Let's go through each component's connections:

- 9V battery: positive to top left, negative to bottom left (net 0).
- R1 (470) is between top left node (let's call it V_top_left) and a node connected to L1 and Q1's collector? Wait, no, looking at the schematic:

Nodes:

1. Top left node (connected to 9V+ and R1 top, R2 top, R3 top, R4 top)
2. Node under R1, above L1 and Q1's base? Wait, no, let's look at the connections:

- R1 (470) is between top node (let's call it N1) and a node connected to L1 (LED) and Q1's base? Wait, no, the LED L1 is connected